# Session 7 notebook: building portfolios from risk alone

**Question.** Markowitz tells you what to hold once you know expected returns (&mu;) and the covariance matrix (&Sigma;): the weights are proportional to &Sigma;<sup>&minus;1</sup>&mu;. Week 3 showed that &mu; is the part we cannot estimate. What can you build from &Sigma; alone, and what are you assuming about &mu; when you do?

| Part | What you do | When |
|---|---|---|
| 1 | Markowitz on five industries: how much do the weights move when &mu; moves by one standard error? | before class, 10 min |
| 2 | Five portfolios from one &Sigma;, on Week 3's 60 characteristics | in class |
| 3 | Risk parity: stocks and bonds since 1958 | in class |
| 4 | Discuss | in class |

**Data.** Parts 1 and 3 download free data straight from Ken French's library and from FRED. Part 2 reuses `week3_data.zip` from Week 3 (the link is on the course notes page).

Cells marked **Your turn** work as written. Change the value, rerun, and say what changed and why.

## Part 0. Setup (before class)

In [ ]:
import io, re, zipfile, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
pd.set_option("display.width", 160, "display.max_columns", 30, "display.float_format", "{:,.2f}".format)

KF = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"

def kf(name, table=0):
    """Read one table from a Ken French CSV zip. Returns decimals, indexed by month ('YYYY-MM') or by day."""
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(KF + name + "_CSV.zip").read()))
    lines = z.read(z.namelist()[0]).decode("latin1").splitlines()
    i = [k for k, l in enumerate(lines) if l.startswith(",")][table]
    rows = []
    for l in lines[i + 1:]:
        if not l.strip() or not l.strip()[0].isdigit():
            break
        rows.append(l)
    df = pd.read_csv(io.StringIO("\n".join([lines[i]] + rows)), index_col=0)
    df.columns = [c.strip() for c in df.columns]
    idx = df.index.astype(str).str.strip()
    df.index = pd.to_datetime(idx, format="%Y%m%d") if len(idx[0]) == 8 else (idx.str[:4] + "-" + idx.str[4:])
    return df.replace([-99.99, -999], np.nan) / 100

def bond10y():
    """Monthly return on a 10-year Treasury bought at par, from FRED's GS10 yield. An approximation:
    it ignores that the bond ages by a month, which barely matters for a 10-year bond."""
    y = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=GS10", index_col=0)["GS10"] / 100
    y.index = y.index.str[:7]
    y0, y1, n = y.shift(1), y, 20                          # last month's coupon, this month's yield, 20 half-years
    price = y0 / y1 * (1 - (1 + y1 / 2) ** -n) + (1 + y1 / 2) ** -n
    return (price - 1 + y0 / 12).dropna().rename("bond")

def sharpe(r): return r.mean() / r.std() * np.sqrt(12)

ff = kf("F-F_Research_Data_Factors")
ind = kf("5_Industry_Portfolios")
print("Ken French factors:", ff.index[0], "to", ff.index[-1], "| industries:", list(ind.columns))

## Part 1. Markowitz on five industries (before class)

Reading #11 asks: given what you believe about expected returns, variances and covariances, which portfolio should you hold? With a risk-free asset the answer is the *tangency* portfolio, weights &prop; &Sigma;<sup>&minus;1</sup>&mu;. Here the assets are Ken French's five industry portfolios, 1963&ndash;2025.

Two portfolios:
- **Tangency**, &Sigma;<sup>&minus;1</sup>&mu;: uses both estimates.
- **Minimum variance**, &Sigma;<sup>&minus;1</sup>1: uses only &Sigma;. It is the tangency portfolio *if every industry has the same expected return*.

Both are scaled so the weights sum to 1.

In [ ]:
X = ind.loc["1963-07":"2025-12"].sub(ff["RF"], axis=0).dropna()       # excess returns
mu, S, T = X.mean(), X.cov(), len(X)
se = X.std() / np.sqrt(T)                                              # standard error of each mean

def tangency(mu, S):
    w = np.linalg.solve(S, mu); return w / w.sum()
def min_variance(S):
    w = np.linalg.solve(S, np.ones(len(S))); return w / w.sum()

print(f"{T} months. Average excess return and its standard error, % a year:")
print(pd.DataFrame({"mean": mu * 1200, "std. error": se * 1200}).T.round(1))
pd.DataFrame({"tangency": tangency(mu, S), "minimum variance": min_variance(S)}, index=X.columns).T.round(2)

The five average returns are within about one standard error of each other. Now redraw &mu; 1,000 times, each time moving every industry's mean by a random amount of about one standard error (a plausible alternative history), and recompute the tangency weights.

In [ ]:
NOISE = 1.0        # Your turn: try 0.5, then 0.25 (how precisely would you need to know mu?)
rng = np.random.default_rng(0)
draws = pd.DataFrame([tangency(mu + NOISE * rng.normal(0, 1, len(mu)) * se, S) for _ in range(1000)], columns=X.columns)
print("Tangency weights across the 1,000 redraws (5th, 50th and 95th percentiles):")
print(draws.quantile([0.05, 0.5, 0.95]).round(2))
draws.clip(-3, 3).boxplot(figsize=(8, 3.5), whis=(5, 95), showfliers=False)
plt.axhline(0, color="grey", lw=0.8); plt.title("Tangency weights when mu moves by about one standard error"); plt.ylabel("weight"); plt.show()

The minimum-variance portfolio doesn't use &mu;, so the redraws don't move it at all. That isn't the same as saying it is stable: &Sigma; is estimated too. Compare the two halves of the sample.

In [ ]:
halves = {}
for a, b in [("1963-07", "1994-06"), ("1994-07", "2025-12")]:
    Y = X.loc[a:b]
    halves[(f"{a[:4]}-{b[:4]}", "tangency")] = tangency(Y.mean(), Y.cov())
    halves[(f"{a[:4]}-{b[:4]}", "min variance")] = min_variance(Y.cov())
pd.DataFrame(halves, index=X.columns).T.round(2)

**Check yourself.**
1. A one-standard-error change in &mu; is something you could never rule out from the data. How far does it move the tangency weights?
2. Which portfolio changed more between the two halves of the sample? Why would that be?
3. Markowitz wrote that the inputs should be *beliefs*, not just past averages. Given what you just saw, why does that matter?

## Part 2. Five portfolios from one &Sigma; (in class)

Back to Week 3's data: 60 characteristic portfolios (8 Barra styles and 52 Chen&ndash;Zimmermann signals), market exposure removed. Estimate on 2015&ndash;2020, test on 2021&ndash;2026, exactly as in Session 5 Part 4.

Every portfolio below is &Sigma;<sup>&minus;1</sup>&mu; with a different assumption (a *prior*) about &mu; or &Sigma;:

| Portfolio | Weights | Same as &Sigma;<sup>&minus;1</sup>&mu; if... |
|---|---|---|
| Session 5's optimal mix | &Sigma;<sup>&minus;1</sup>&mu;&#770; | you trust the sample means |
| Session 5's ridge | (&Sigma; + &gamma;I)<sup>&minus;1</sup>&mu;&#770; | you shrink toward "no near-arbitrage" (KNS) |
| Minimum variance | &Sigma;<sup>&minus;1</sup>1 | every signal earns the same expected return |
| Inverse volatility | 1/&sigma;<sub>i</sub> | every signal has the same Sharpe ratio, and they are uncorrelated |
| Equal weight | 1/N | every signal has the same mean and variance, and they are uncorrelated |

The last three don't use the sample means at all. They use the **sign** of each signal, which Chen and Zimmermann set from the original paper: a prior that comes from decades of earlier data.

In [ ]:
# The class data (week3_data.zip). The link is on the course notes page, not in this public notebook.
DATA_ZIP_LINK = ""     # <- paste the Google Drive link to week3_data.zip between the quotes, then run this cell

DATA_DIR = Path("data")
if not (DATA_DIR / "w3_panel.parquet").exists():
    fid = re.search(r"/d/([\w-]+)|[?&]id=([\w-]+)", DATA_ZIP_LINK)
    assert fid, "Paste the week3_data.zip link from the course notes page into DATA_ZIP_LINK above, then rerun this cell."
    fid = fid.group(1) or fid.group(2)
    urllib.request.urlretrieve(f"https://drive.usercontent.google.com/download?id={fid}&export=download&confirm=t", "week3_data.zip")
    assert zipfile.is_zipfile("week3_data.zip"), "The download did not return a zip. Check that the link is shared with you, and try again."
    zipfile.ZipFile("week3_data.zip").extractall(".")

panel = pd.read_parquet(DATA_DIR / "w3_panel.parquet")
ffm = pd.read_parquet(DATA_DIR / "ff_factors_monthly.parquet").set_index("month")
signals = pd.read_csv(DATA_DIR / "w3_signals.csv")
STYLES = ["size", "value", "momentum", "beta", "resid_vol", "leverage", "liquidity", "profitability"]
CHARS = STYLES + signals["acronym"].tolist()

# Same construction as Session 5: z-score each characteristic each month, form characteristic-weighted portfolios,
# then remove each portfolio's market exposure.
panel[CHARS] = panel.groupby("month")[CHARS].transform(lambda x: (x - x.mean()) / x.std()).astype("float32")
F = panel.groupby("month").apply(lambda g: pd.Series(g[CHARS].values.T @ g["y"].values / len(g), index=CHARS),
                                 include_groups=False)
mkt = ffm.loc[F.index, "mktrf"].values
for c in CHARS:
    F[c] = F[c] - np.polyfit(mkt, F[c].values, 1)[0] * mkt
print(f"{F.shape[1]} portfolios, {F.index[0]} to {F.index[-1]}")

In [ ]:
SPLIT = "2021-01"      # Your turn: try "2019-01" or "2023-01"
train = F.index < SPLIT
A, B = F[train], F[~train]
m, V = A.mean().values, A.cov().values
N, sd = len(m), np.sqrt(np.diag(V))

weights = {
    "Session 5 optimal mix": np.linalg.solve(V, m),
    "Session 5 ridge": np.linalg.solve(V + 1.58 * np.trace(V) * np.eye(N), m),
    "minimum variance": np.linalg.solve(V, np.ones(N)),
    "inverse volatility": 1 / sd,
    "equal weight": np.ones(N),
}
rows = {}
for name, w in weights.items():
    rows[name] = {"Sharpe in sample": sharpe(A.values @ w), "Sharpe out of sample": sharpe(B.values @ w),
                  "share of short positions": (w < 0).mean()}
print(f"Train {A.index[0]} to {A.index[-1]} ({len(A)} months); test {B.index[0]} to {B.index[-1]} ({len(B)} months)")
print(f"Signals with a negative average return in the training sample: {(m < 0).sum()} of {N}")
pd.DataFrame(rows).T.round(2)

Look at the minimum-variance row. It ignores &mu;, yet it does badly. Its weights still go through &Sigma;<sup>&minus;1</sup>, and Session 5 showed what &Sigma;<sup>&minus;1</sup> does with near-twins: large, offsetting bets on the gap between them. A prior on &mu; is not enough; the estimate of &Sigma; needs taming too. One fix is to shrink &Sigma; toward its diagonal (pretend the signals are less correlated than they look).

In [ ]:
SHRINK = 0.5       # Your turn: 0 is the sample covariance, 1 ignores all correlations (then min variance is 1/variance)
V_shrunk = (1 - SHRINK) * V + SHRINK * np.diag(np.diag(V))
w = np.linalg.solve(V_shrunk, np.ones(N))
print(f"minimum variance with shrunk Sigma: Sharpe in sample {sharpe(A.values @ w):.2f}, out of sample {sharpe(B.values @ w):.2f}")

**Check yourself.**
1. Inverse volatility and equal weight did better out of sample than in sample. Is that evidence they are good, or that 2021&ndash;2026 was a good period for these signals? (Try a different `SPLIT`.)
2. Why can equal weight work when a third or more of the signals had negative average returns in the training period?
3. Ridge, inverse volatility and shrinking &Sigma; are all ways of *not* trusting the sample. What is each one not trusting?

## Part 3. Risk parity: stocks and bonds since 1958 (in class)

A 60/40 portfolio looks balanced in dollars. In risk it is not: stocks are about two and a half times as volatile as 10-year Treasuries, so they supply almost all of the portfolio's variance. Risk parity balances *risk* instead: hold each asset in proportion to 1/&sigma;, then borrow to bring the total volatility back up to what the investor wants.

We build three portfolios each month, using only data available at the time (trailing 36-month volatilities):
- **60/40**: 60% stocks, 40% bonds.
- **Inverse volatility**: weights &prop; 1/&sigma;, no leverage.
- **Risk parity (levered)**: inverse volatility, levered to 60/40's trailing volatility, paying T-bill + 0.5% a year on the borrowed amount.

With two assets, inverse volatility and "equal risk contribution" are the same portfolio.

In [ ]:
BORROW_SPREAD = 0.005      # Your turn: 0, 0.01. Who borrows at T-bills + 0.5%?
bond = bond10y()
D = pd.concat([ff["Mkt-RF"].rename("stocks"), bond.sub(ff["RF"]).rename("bonds"), ff["RF"]], axis=1).dropna().loc["1955-01":]

vol = D[["stocks", "bonds"]].rolling(36).std().shift(1)                 # known at the start of the month
iv = (1 / vol).div((1 / vol).sum(axis=1), axis=0)
r6040 = 0.6 * D.stocks + 0.4 * D.bonds
riv = (iv * D[["stocks", "bonds"]]).sum(axis=1)
lev = (r6040.rolling(36).std() / riv.rolling(36).std()).shift(1)
rrp = lev * riv - (lev - 1).clip(lower=0) * BORROW_SPREAD / 12
R = pd.concat({"stocks": D.stocks, "bonds": D.bonds, "60/40": r6040, "inverse vol": riv, "risk parity": rrp}, axis=1).dropna()

def max_drawdown(r):
    wealth = (1 + r + D.RF.reindex(r.index)).cumprod()
    return (wealth / wealth.cummax() - 1).min()

share = (0.6**2 * D.stocks.var() + 0.6 * 0.4 * D.stocks.cov(D.bonds)) / r6040.var()
print(f"{R.index[0]} to {R.index[-1]}. Share of 60/40's variance that comes from stocks: {share:.0%}")
print(f"Average stock weight in inverse vol: {iv.stocks.mean():.0%}; average leverage of risk parity: {lev.reindex(R.index).mean():.2f}x")
pd.DataFrame({"excess return": R.mean() * 12, "volatility": R.std() * np.sqrt(12),
              "Sharpe": R.apply(sharpe), "max drawdown": R.apply(max_drawdown)}).round(2)

Risk parity's case rests on bonds earning at least as much per unit of risk as stocks (Asness, Frazzini and Pedersen). Did they? Split the sample at the peak in interest rates (September 1981) and at the end of 2020.

In [ ]:
periods = {"1958-1981 (rates rising)": ("1958-01", "1981-09"),
           "1981-2020 (rates falling)": ("1981-10", "2020-12"),
           "2021-today": ("2021-01", R.index[-1])}
pd.DataFrame({k: R.loc[a:b].apply(sharpe) for k, (a, b) in periods.items()}).T.round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
D.stocks.rolling(36).corr(D.bonds).plot(ax=ax[0], title="Stock-bond correlation, trailing 36 months")
ax[0].axhline(0, color="grey", lw=0.8)
y22 = R.loc["2022-01":"2022-12"].add(D.RF.loc["2022-01":"2022-12"], axis=0)
((1 + y22).prod() - 1).drop(["bonds"]).plot.bar(ax=ax[1], title="Total return in 2022")
ax[1].axhline(0, color="grey", lw=0.8); plt.tight_layout(); plt.show()

**Check yourself.**
1. Inverse volatility assumes stocks and bonds have the same Sharpe ratio. In which period was that close to true?
2. Why did levered risk parity do worse than 60/40 in 2022? Name the two things that went wrong at once.
3. Set `BORROW_SPREAD` to 0. How much of the gap with 60/40 is the cost of leverage?
4. These are US stocks and one bond. Real risk parity funds hold many asset classes and countries. Why would that help?

## Part 4. Discuss (in class)

1. Each portfolio in this notebook is &Sigma;<sup>&minus;1</sup>&mu; with a prior. For minimum variance, inverse volatility, 1/N and risk parity, state the prior in one sentence, and give a period when it was wrong.
2. Asness, Frazzini and Pedersen argue that investors who cannot or will not borrow overpay for risky assets, so safe assets earn more per unit of risk. Who are those investors? What would happen to the risk parity premium if they started borrowing?
3. In Part 3, 60/40 lost about 18% in 2022 and levered risk parity lost more. A pension fund asks: should it move toward risk parity, away from it, or neither? What would you want to know first?
4. Wednesday is about trading costs. Which portfolio in this notebook would cost the most to run, and why?